Meridian - Data Quality Review (Intern 2)
Run this on the RAW monthly files. It prints PASS / FAIL / WARN for each check.

How to use:
1. Put this file in the same folder as your monthly CSV files
   (or change FOLDER below to the folder path).
2. Run:  python data_quality_review.py
   (In Jupyter you can paste each STEP into its own cell.)

In [ ]:
from pathlib import Path
import re
import pandas as pd

FOLDER = Path(".")          # folder that contains the monthly files
MIN_CLAIMS_FOR_PROVIDER = 20  # ignore providers with very few claims


def result(ok, message, level="FAIL"):
    """Print one line: PASS if ok, otherwise FAIL (or WARN)."""
    print(("PASS  " if ok else f"{level}  ") + message)


# Some months use different column names. Map them all to one standard name.
RENAME = {
    "mbr_id": "member_id",
    "svc_from_dt": "service_from_date",
    "svc_to_dt": "service_to_date",
    "allowed_amt": "total_allowed",
    "paid_amt": "total_paid",
}
DATE_COLS = ["service_from_date", "service_to_date", "received_date", "processed_date"]


def find_files(prefix):
    """Find raw monthly files, skip cleaned copies and Jupyter checkpoint copies."""
    files = []
    for f in sorted(FOLDER.rglob(f"{prefix}_*.csv")):
        if "_cleaned" in f.name or ".ipynb_checkpoints" in f.parts or "checkpoint" in f.name:
            continue
        files.append(f)
    return files


def month_of(file):
    """claims_header_2026_01.csv -> '2026_01'"""
    return re.search(r"(\d{4}_\d{2})", file.name).group(1)


def parse_dates(series):
    """Some months are YYYY-MM-DD, some are DD-MM-YYYY. Detect and parse each."""
    s = series.dropna().astype(str)
    if len(s) and s.str.match(r"^\d{4}-\d{2}-\d{2}$").all():
        return pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
    return pd.to_datetime(series, format="%d-%m-%Y", errors="coerce")


# ---------------------------------------------------------------- STEP 1
print("\n===== STEP 1: Which files did we find? =====")
header_files = find_files("claims_header")
line_files = find_files("claim_lines")
premium_files = find_files("premiums")
print("header files :", [f.name for f in header_files])
print("line files   :", [f.name for f in line_files])
print("premium files:", [f.name for f in premium_files])
result(len(header_files) == len(line_files) == len(premium_files),
       "Same number of header, line and premium files")

# ---------------------------------------------------------------- STEP 2
print("\n===== STEP 2: Do all months have the same columns? =====")
headers = {}
for f in header_files:
    headers[month_of(f)] = pd.read_csv(f, dtype=str)

base_cols = set(next(iter(headers.values())).columns)
for month, df in headers.items():
    cols = set(df.columns)
    extra, missing = cols - base_cols, base_cols - cols
    ok = not extra and not missing
    result(ok, f"{month}: {df.shape[0]} rows, {df.shape[1]} columns"
               + ("" if ok else f" | new: {sorted(extra)} | missing: {sorted(missing)}"))

# Now fix the names, convert types, and stack all months into one table
frames = []
for month, df in headers.items():
    df = df.rename(columns=RENAME)
    for c in DATE_COLS:
        df[c] = parse_dates(df[c])
    for c in ["total_billed", "total_allowed", "total_paid", "member_responsibility"]:
        df[c] = pd.to_numeric(df[c])
    df["feed"] = month
    frames.append(df)
H = pd.concat(frames, ignore_index=True)

lines = []
for f in line_files:
    d = pd.read_csv(f, dtype={"procedure_code": str})
    d["feed"] = month_of(f)
    lines.append(d)
L = pd.concat(lines, ignore_index=True)

P = pd.concat([pd.read_csv(f) for f in premium_files], ignore_index=True)

# ---------------------------------------------------------------- STEP 3
print("\n===== STEP 3: Duplicates =====")
data_cols = [c for c in H.columns if c != "feed"]
exact = H.duplicated(data_cols).sum()
result(exact == 0, f"Exact duplicate header rows: {exact}", "WARN")
H = H.drop_duplicates(data_cols)

repeat = H.claim_id.duplicated().sum()
result(repeat == 0,
       f"claim_ids that appear in more than one month: {repeat} "
       "(pending claims are re-sent with a new status)", "WARN")

# Keep only the LATEST version of every claim
latest = H.sort_values("feed").groupby("claim_id").tail(1)
print(f"Unique claims (latest version): {len(latest)}  |  status counts:",
      latest.claim_status.value_counts().to_dict())

# Same idea for lines
L_latest = L.sort_values("feed").groupby(["claim_id", "line_number"]).tail(1)
print(f"Claim line rows: {len(L)}  ->  unique lines: {len(L_latest)}")

# ---------------------------------------------------------------- STEP 4
print("\n===== STEP 4: Business rules =====")
paid = latest[latest.claim_status == "Paid"]
denied = latest[latest.claim_status == "Denied"]

result(((denied.denial_code.isna()).sum() == 0), "Every Denied claim has a denial code")
result(((latest.claim_status != "Denied") & latest.denial_code.notna()).sum() == 0,
       "No denial code on non-denied claims")
result((paid.total_paid <= 0).sum() == 0, "Every Paid claim has paid amount > 0")
bad = ((paid.total_paid + paid.member_responsibility - paid.total_allowed).abs() > 0.02).sum()
result(bad == 0, f"Paid + member responsibility = allowed  (failures: {bad})")

normal = latest[latest.claim_status != "Adjusted"]  # Adjusted claims are negative on purpose
result((normal.total_billed < 0).sum() == 0, "No negative billed amounts (except Adjusted)")
result((normal.total_paid > normal.total_billed).sum() == 0, "Paid is never more than billed")

early = latest[latest.received_date < latest.service_from_date]
result(len(early) == 0, f"Claim received BEFORE the service date: {len(early)}")
if len(early):
    print("   by provider:", early.provider_id.value_counts().to_dict())
late = latest[latest.processed_date < latest.received_date]
result(len(late) == 0, f"Claim processed BEFORE it was received: {len(late)}")

# header vs lines
orphan_lines = (~L.claim_id.isin(H.claim_id)).sum()
no_lines = (~H.claim_id.isin(L.claim_id)).sum()
result(orphan_lines == 0, f"Claim lines with no header: {orphan_lines}")
result(no_lines == 0, f"Headers with no claim lines: {no_lines}")

sums = L.groupby(["claim_id", "feed"])[["billed_amount", "paid_amount"]].sum().reset_index()
J = H.merge(sums, on=["claim_id", "feed"])
diff_billed = ((J.total_billed - J.billed_amount).abs() > 0.05).sum()
diff_paid = ((J.total_paid - J.paid_amount).abs() > 0.05).sum()
result(diff_billed == 0, f"Header total_billed = sum of lines  (failures: {diff_billed})")
result(diff_paid == 0, f"Header total_paid = sum of lines  (failures: {diff_paid})")

# premiums
dup_prem = P.duplicated(["member_id", "premium_month"]).sum()
result(dup_prem == 0, f"Duplicate member + month in premiums: {dup_prem}")
missing_members = (~latest.member_id.isin(P.member_id)).sum()
result(missing_members == 0, f"Claim members that are not in premiums: {missing_members}", "WARN")

# possible duplicate billing: same member, provider, date and diagnosis
key = ["member_id", "provider_id", "service_from_date", "primary_diagnosis"]
dups = normal.duplicated(key, keep=False).sum()
result(dups == 0, f"Claims sharing member+provider+date+diagnosis: {dups}", "WARN")

# ---------------------------------------------------------------- STEP 5
print("\n===== STEP 5: Provider outliers (compared with similar providers) =====")
x = latest[latest.claim_status.isin(["Paid", "Denied", "Pending"])]
g = x.groupby(["place_of_service", "provider_id"]).agg(
    claims=("claim_id", "count"),
    avg_billed=("total_billed", "mean"),
    total_paid=("total_paid", "sum"),
    denial_rate=("claim_status", lambda s: (s == "Denied").mean()),
).reset_index()
g = g[g.claims >= MIN_CLAIMS_FOR_PROVIDER]

# z-score = how many standard deviations away from providers in the same group
for col in ["avg_billed", "denial_rate"]:
    g["z_" + col] = g.groupby("place_of_service")[col].transform(
        lambda s: (s - s.mean()) / s.std())

print("\nHighest average billed amount vs peers:")
print(g.sort_values("z_avg_billed", ascending=False).head(8).round(2).to_string(index=False))
print("\nHighest denial rate vs peers:")
print(g.sort_values("z_denial_rate", ascending=False).head(8).round(2).to_string(index=False))

print("\nDone. Every FAIL / WARN above is something to explain or fix.")